Feature engineering and removal of reduntant values

In [1]:

import pandas as pd
df= pd.read_csv("train.csv")
df.shape
# DROPPED COLUMNS AND REASONS
dropped = {
    'PoolQC': 'only 3 houses have pool, not enough data',
    'RoofMatl': '99% same value, no variation',
    'GarageCond': 'redundant with GarageQual, lower variation',
    'Alley': 'only 91 houses have alley access, not enough data',
    'Utilities': '99% same value, no variation',
    'Heating': '99% same value, no variation',
    'LowQualFinSF': 'limited data availability',
    'PoolArea': 'limited data availability',
    'MiscFeature': 'lot of missing values ',
    'MiscVal': 'limited data availability',
    'LandSlope': '99% same value, no variation',
    'BsmtHalfBath': 'limited data availability',
    'BsmtFinSF2': 'limited data availability',
    'BsmtFinType2': '99% same value, no variation',
    'BsmtUnfSF': 'limited data availability',
    'KitchenAbvGr': '99% same value, no variation',
    'GarageCond': 'redundant with GarageQual, lower variation',
    'MoSold': ' it do not affect the price of the house',
    'YrSold': ' it do not affect the price of the house',
    'OpenPorchSF': 'combined with other porch features to create TotalPorchSF',
    'EnclosedPorch': 'combined with other porch features to create TotalPorchSF',
    '3SsnPorch': 'combined with other porch features to create TotalPorchSF',
    'ScreenPorch': 'combined with other porch features to create TotalPorchSF',
    'BsmtFinSF1':'already have total basement sf',
    'BsmtFinSF2': 'already have total basement sf',
    'BsmtUnfSF': 'already have total basement sf',
    'OverallCond': ' less impact on price compared to OverallQuall, corr in negative',
    
    
}

df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
df.shape
df = df.drop(columns=list(dropped.keys()), errors='ignore')
df.shape
df['MSSubClass'] = df['MSSubClass'].astype(str)
df['Electrical'].value_counts()
df['Electrical'].dtype


df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])
# Filling garage-related missing values

garage_cols = ["GarageType", "GarageFinish", "GarageQual"]

for col in garage_cols:
    df[col] = df[col].fillna("NA")

df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
df['BsmtExposure'].value_counts()
df['BsmtQual'].value_counts()
pd.crosstab(df["BsmtQual"], df["BsmtCond"])
df.groupby("BsmtQual")["SalePrice"].mean()
df.groupby("BsmtCond")["SalePrice"].mean()
df.loc[948, "BsmtExposure"] = "No"
# Filling basement-related missing values

bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtFinType1", "BsmtExposure"]

for col in bsmt_cols:
    df[col] = df[col].fillna("NA")
df['FireplaceQu']= df['FireplaceQu'].fillna("NA")
df['Fence']= df['Fence'].fillna("NA")
mode_value = df["MasVnrType"].mode()[0]

df.loc[[624, 1300, 1334], "MasVnrType"] = mode_value
# Filling remaining missing masonry veneer values

df["MasVnrType"] = df["MasVnrType"].fillna("None")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0.0)
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print(missing_values)


df = df.drop(columns=['1stFlrSF', '2ndFlrSF'])
df.shape
df['TotalBath'] = df['FullBath'] + df['HalfBath'] + df['BsmtFullBath']

df=df.drop(columns=['FullBath', 'HalfBath', 'BsmtFullBath'])









Series([], dtype: int64)


Removed outliers from saleprice

applying log to decrease skweness

In [2]:
import numpy as np
df['LotArea'] = np.log1p(df['LotArea'])
df['MasVnrArea']= np.log1p(df['MasVnrArea'])
df['GrLivArea']= np.log1p(df['GrLivArea'])
df['WoodDeckSF']= np.log1p(df['WoodDeckSF'])
df['LotFrontage']= np.log1p(df['LotFrontage'])

In [3]:
df = pd.get_dummies(df)
print(df.shape)

(1460, 256)


In [4]:
from sklearn.model_selection import train_test_split
X=df.drop(columns=['SalePrice']) # This is DATA
y=df['SalePrice'] # This is TARGET

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.25, random_state=42)



In [5]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np


y_pred= model.predict(X_test)

rmse=np.sqrt(mean_squared_error(y_test, y_pred))
r2= r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100




print(f"RMSE: {rmse:,.0f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.1f}%")
print(f"MAE: {mae:.4f}")

RMSE: 32,864
R2 Score: 0.8458
MAPE: 12.9%
MAE: 20977.8046


Scaling

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred=model.predict(X_test_scaled)

rmse=np.sqrt(mean_squared_error(y_test, y_pred))
r2= r2_score(y_test, y_pred)  
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100  

print(f"RMSE: {rmse:,.0f}")
print(f"R2 Score: {r2:.4f}")
print(f"MAPE: {mape:.1f}%")
print(f"MAE: {mae:.4f}")

RMSE: 32,719
R2 Score: 0.8472
MAPE: 12.9%
MAE: 20892.5100


In [8]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
xgb.fit(X_train, y_train)

preds = xgb.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))
print("R2:", r2_score(y_test, preds))
print("MAE:", mean_absolute_error(y_test, preds))
print("MAPE:", np.mean(np.abs((y_test - preds) / y_test))*100, "%")

RMSE: 25849.36053367665
R2: 0.9046167135238647
MAE: 16664.759765625
MAPE: 10.050354849123913 %


sale price still has skewness

In [9]:
X=df.drop(columns=['SalePrice']) # This is DATA
y=np.log1p(df['SalePrice']) # This is TARGET

In [10]:
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.25, random_state=42)

from sklearn.preprocessing import StandardScaler 

scaler = StandardScaler() 

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

model = LinearRegression() 
model.fit(X_train_scaled, y_train)

y_pred= model.predict(X_test_scaled)

pred_actual= np.expm1(y_pred) 
test_actual= np.expm1(y_test) 

rmse=np.sqrt(mean_squared_error(test_actual, pred_actual)) 
r2= r2_score(test_actual, pred_actual) 
print(f"RMSE: {rmse:,.0f}")
print(f"R2 Score: {r2:.4f}")
print("MAE:", mean_absolute_error(test_actual, pred_actual))
print("MAPE:", np.mean(np.abs((test_actual - pred_actual) / test_actual))*100, "%")

RMSE: 27,192
R2 Score: 0.8945
MAE: 16615.719709365658
MAPE: 9.588013858688814 %
